In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

In [3]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.load_local(
    "./faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

In [4]:
vectorstore

In [5]:
retriever = vectorstore.as_retriever()

In [6]:
from langchain.tools import tool

@tool
def search_documents(query: str) -> str:
    """2026년 테크노빌드 주식회사 임직원 통합 가이드북 검색기입니다."""
    docs = retriever.invoke(query)
    return "\n\n".join([doc.page_content for doc in docs])

In [7]:
system_prompt = """당신은 테크노빌드 주식회사 가이드북 정보를 친절하게 제공하는 어시스턴트입니다.

1. 정보가 필요할 경우 반드시 검색 도구(search_documents)를 사용하여 확인하세요.
2. 답변은 반드시 검색된 문서의 내용에만 기반하여 작성하세요.
3. 문서에 관련 내용이 없다면 추측하지 말고 모른다고 답변하세요.
"""

In [8]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=[search_documents],
    system_prompt=system_prompt
)

In [9]:
from langchain.messages import HumanMessage

query = "국가 기술 자격 중 기사 자격증을 취득하면 얼마를 받을 수 있을까?"

response = agent.invoke({
    "messages": [HumanMessage(content=query)]
})

response

{'messages': [HumanMessage(content='국가 기술 자격 중 기사 자격증을 취득하면 얼마를 받을 수 있을까?', additional_kwargs={}, response_metadata={}, id='3f74460d-f984-4f71-94bf-21dee376f0d5'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_documents', 'arguments': '{"query": "\\uae30\\uc0ac \\uc790\\uaca9\\uc99d \\uc218\\ub2f9"}'}, '__gemini_function_call_thought_signatures__': {'a6096e0e-937d-4707-bdba-dfeed73cf93d': 'EjQKMgEMOdbHLAlAN2u+EXaHSHEwwDyjqWmUPnatXqMAPj2CecT94MZS27gzjAxpoR5cAxsI'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e2ff9-50d3-71b1-9323-d66d71508c32-0', tool_calls=[{'name': 'search_documents', 'args': {'query': '기사 자격증 수당'}, 'id': 'a6096e0e-937d-4707-bdba-dfeed73cf93d', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 170, 'output_tokens': 22, 'total_tokens': 192, 'input_token_details': {'cache_read': 0}}),
  ToolMessage(cont